# Partner belief analysis

A companion to `belief_analysis.ipynb`, split out because it asks a distinct question: not whether a participant's *own* stated belief (`collabBelief`) predicts their own strategy choice (established there), but whether their *partner's* stated belief does. This is a natural place to expect an `arm` interaction rather than a main effect: a participant's belief can only be shown to their partner through the robot's info modal (`"Partner Belief = " + partnerCollabBelief`), a treatment-only mechanism. In control, there's no channel at all for `partner_belief` to reach the other participant -- so `partner_belief` should have no real relationship with a control participant's choice, and any relationship in treatment would be attributable to that sharing mechanism (or, more precisely, to `arm` broadly -- this can't isolate whether a participant actually clicked the robot *that specific* round, only whether they were in the arm where doing so was possible).

Both models from `belief_analysis.ipynb` are reproduced here as controls -- the `max_difficulty_c`/`diff_difficulty_c`/`own_magnitude_c` version and the `R_c`/`diff_u_c`/`own_magnitude_c` sensitivity-check version -- so `partner_belief_c` is tested against each, exactly as it depends on `belief_analysis.ipynb`'s established own-belief results without re-deriving that notebook's full methodological build-up (see it for the pair-level tests, the VB-vs-GEE caveat, and the interaction-testing narrative behind these formulas).

## Setup: reproduce the reshaped data and covariates from `belief_analysis.ipynb`

Same construction as there, condensed into one cell -- see that notebook for the reasoning behind each covariate.

In [1]:
import json

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

task = pd.read_csv("task_data.csv")
task_summary = pd.read_csv("task_summary.csv")

missing = (task["strategy_1"] == "undefined") | (task["strategy_2"] == "undefined")
task = task[~missing].copy()
task["pair_id"] = task["username_1"] + "_" + task["username_2"]

real_tasks = task_summary[task_summary["task_difficulty"].notna()].copy()
difficulty_by_index = real_tasks.set_index("task_index")["task_difficulty"]
magnitude_by_index = real_tasks.set_index("task_index")["payoff_magnitude"]
difficulty_1 = task["task_1"].map(difficulty_by_index).astype(int)
difficulty_2 = task["task_2"].map(difficulty_by_index).astype(int)
task["max_difficulty"] = np.maximum(difficulty_1, difficulty_2)
task["diff_difficulty"] = (difficulty_1 - difficulty_2).abs()

# risk-dominance covariates (u of the actually-selected design), as in
# outcome_analysis_selected_u.ipynb
with open("../data/experiment.json", encoding="utf-8") as f:
    experiment = json.load(f)

COLLAB_LABELS = ["Design K", "Design L", "Design M"]
rank_by_task = {}
for task_index in range(30):
    options_by_label = {option["label"]: option for option in experiment["tasks"][task_index]["options"]}
    ranked = sorted(COLLAB_LABELS, key=lambda label: int(options_by_label[label]["upside"]), reverse=True)
    rank_by_task[task_index] = {
        label.replace("Design ", ""): rank_letter
        for rank_letter, label in zip(["A", "B", "C"], ranked)
    }


def compute_u(design):
    v_ci = real_tasks[f"V_{design}_CI"]
    v_cc = real_tasks[f"V_{design}_CC"]
    return (real_tasks["V_Y_II"] - v_ci) / ((real_tasks["V_Y_II"] - v_ci) + (v_cc - real_tasks["V_Y_IC"]))


real_tasks["u_A"] = compute_u("A")
real_tasks["u_B"] = compute_u("B")
real_tasks["u_C"] = compute_u("C")
u_by_task_and_rank = real_tasks.set_index("task_index")[["u_A", "u_B", "u_C"]].to_dict("index")


def u_selected(task_index, design):
    if design == "Y":
        return u_by_task_and_rank[task_index]["u_A"]
    rank_letter = rank_by_task[task_index][design]
    return u_by_task_and_rank[task_index][f"u_{rank_letter}"]


task["u_selected_1"] = task.apply(lambda r: u_selected(r["task_1"], r["design_1"]), axis=1)
task["u_selected_2"] = task.apply(lambda r: u_selected(r["task_2"], r["design_2"]), axis=1)
task["diff_u"] = (task["u_selected_1"] - task["u_selected_2"]).abs()


def logit(p):
    return np.log(p / (1 - p))


task["R"] = 0.5 * logit(task["u_selected_1"]) + 0.5 * logit(task["u_selected_2"])

shared_cols = ["arm", "pair_id", "round", "max_difficulty", "diff_difficulty", "diff_u", "R"]
belief = pd.concat([
    task[shared_cols + ["username_1", "task_1", "collabBelief_1", "collabBelief_2", "strategy_1"]].rename(
        columns={"username_1": "username", "task_1": "task", "collabBelief_1": "collabBelief",
                 "collabBelief_2": "partner_belief", "strategy_1": "strategy"}),
    task[shared_cols + ["username_2", "task_2", "collabBelief_2", "collabBelief_1", "strategy_2"]].rename(
        columns={"username_2": "username", "task_2": "task", "collabBelief_2": "collabBelief",
                 "collabBelief_1": "partner_belief", "strategy_2": "strategy"}),
], ignore_index=True)

belief["chose_C"] = (belief["strategy"] == "C").astype(int)
belief["own_magnitude"] = belief["task"].map(magnitude_by_index)
belief["collabBelief_c"] = belief["collabBelief"] - belief["collabBelief"].mean()
belief["partner_belief_c"] = belief["partner_belief"] - belief["partner_belief"].mean()
belief["max_difficulty_c"] = belief["max_difficulty"] - belief["max_difficulty"].mean()
belief["diff_difficulty_c"] = belief["diff_difficulty"] - belief["diff_difficulty"].mean()
belief["own_magnitude_c"] = belief["own_magnitude"] - belief["own_magnitude"].mean()
belief["diff_u_c"] = belief["diff_u"] - belief["diff_u"].mean()
belief["R_c"] = belief["R"] - belief["R"].mean()

print(f"n={len(belief)} (participant, round) observations, {belief['username'].nunique()} participants, "
      f"{belief['pair_id'].nunique()} pairs")

n=1556 (participant, round) observations, 52 participants, 26 pairs


## Does the partner's belief predict a participant's strategy choice?

Model: `chose_C ~ collabBelief_c + partner_belief_c * arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c + own_magnitude_c + max_difficulty_c:own_magnitude_c` -- own belief, the established difficulty interaction, and the payoff-magnitude terms from `belief_analysis.ipynb` are all kept as controls, and `partner_belief_c` is interacted with `arm` to test exactly this.

In [2]:
partner_formula = (
    "chose_C ~ collabBelief_c + partner_belief_c * arm "
    "+ max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c "
    "+ own_magnitude_c + max_difficulty_c:own_magnitude_c"
)

gee_partner = smf.gee(partner_formula, groups="pair_id", data=belief, family=sm.families.Binomial()).fit()
print(gee_partner.summary())

                               GEE Regression Results                              
Dep. Variable:                     chose_C   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Binomial   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:57:52
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                             

In [3]:
mixed_partner = BinomialBayesMixedGLM.from_formula(
    partner_formula, vc_formulas={"pair": "0 + C(pair_id)"}, data=belief,
).fit_vb()
print(mixed_partner.summary())

                           Binomial Mixed GLM Results
                                   Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------------------------------
Intercept                             M     2.5170   0.0945                      
arm[T.treatment]                      M     1.2954   0.1592                      
collabBelief_c                        M     0.0450   0.0032                      
partner_belief_c                      M     0.0175   0.0029                      
partner_belief_c:arm[T.treatment]     M     0.0123   0.0060                      
max_difficulty_c                      M    -0.6529   0.0842                      
diff_difficulty_c                     M    -0.0952   0.0746                      
arm[T.treatment]:diff_difficulty_c    M     0.5408   0.1307                      
own_magnitude_c                       M    -0.2410   0.0671                      
max_difficulty_c:own_magnitude_c      M    -

### Interpretation

This comes out exactly as the mechanism predicts:

- **`partner_belief_c` (the control-arm slope) is not significant** (GEE coef ≈ 0.010, p = 0.144) -- in control, where partner belief has no way of reaching the other participant, it has no detectable relationship with a participant's own choice, as expected.
- **`partner_belief_c:arm[T.treatment]` is significant** (GEE p = 0.004; the mixed model agrees in sign and magnitude). The effective treatment-arm slope is roughly 0.010 + 0.033 ≈ 0.043 -- comparable in size to the effect of a participant's *own* belief (≈ 0.038-0.040 in `belief_analysis.ipynb`). Only where the belief-sharing mechanism exists does the partner's belief predict behavior.
- `collabBelief_c`, `max_difficulty_c`, `arm:diff_difficulty_c`, `own_magnitude_c`, and `max_difficulty_c:own_magnitude_c` are all essentially unchanged from `belief_analysis.ipynb`'s own-belief model, so this isn't just re-detecting one of those effects under a new name.

This is a nice internal consistency check as much as a result: the one arm where `partner_belief` has a plausible channel to matter is the one arm where it does, and the arm where it structurally cannot matter is the one where it doesn't. That said, this still can't confirm that any given participant actually *saw* their partner's belief that specific round (only that they were in the arm where it was possible) -- so read it as "partner belief matters when the mechanism to know it exists," not as evidence for every individual round.

## Sensitivity check: risk dominance (`u`/`R`) instead of task difficulty

Mirrors `belief_analysis.ipynb`'s own risk-dominance sensitivity check: `partner_belief_c * arm` with `R_c`/`diff_u_c`/`arm:diff_u_c`/`own_magnitude_c` as controls in place of `max_difficulty_c`/`diff_difficulty_c`/`arm:diff_difficulty_c`/`max_difficulty_c:own_magnitude_c`.

In [4]:
partner_u_formula = (
    "chose_C ~ collabBelief_c + partner_belief_c * arm "
    "+ R_c + diff_u_c + arm:diff_u_c + own_magnitude_c"
)

gee_partner_u = smf.gee(partner_u_formula, groups="pair_id", data=belief, family=sm.families.Binomial()).fit()
print(gee_partner_u.summary())

                               GEE Regression Results                              
Dep. Variable:                     chose_C   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Binomial   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:57:53
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                             1.

In [5]:
mixed_partner_u = BinomialBayesMixedGLM.from_formula(
    partner_u_formula, vc_formulas={"pair": "0 + C(pair_id)"}, data=belief,
).fit_vb()
print(mixed_partner_u.summary())

                           Binomial Mixed GLM Results
                                  Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
--------------------------------------------------------------------------------
Intercept                            M     2.4355   0.0926                      
arm[T.treatment]                     M     1.2843   0.1570                      
collabBelief_c                       M     0.0435   0.0031                      
partner_belief_c                     M     0.0178   0.0029                      
partner_belief_c:arm[T.treatment]    M     0.0125   0.0059                      
R_c                                  M    -2.0441   0.2854                      
diff_u_c                             M    -3.6336   0.9897                      
arm[T.treatment]:diff_u_c            M     2.7365   1.4154                      
own_magnitude_c                      M    -0.2451   0.0655                      
pair                                 V     0.9483   0.1

### Interpretation

The internal-consistency check replicates exactly as before:

- **`partner_belief_c` (the control-arm slope) is not significant** (GEE coef 0.011, p = 0.137) -- no detectable relationship in the arm with no sharing mechanism, as expected.
- **`partner_belief_c:arm[T.treatment]` is significant** (GEE coef 0.033, p = 0.004; mixed model agrees in sign and magnitude), giving an effective treatment-arm slope of roughly 0.011 + 0.033 ≈ 0.044 -- essentially identical to the `max_difficulty_c`-based version's ≈0.043, and again comparable to a participant's own belief effect.
- `collabBelief_c`, `R_c`, `diff_u_c`, `arm:diff_u_c`, and `own_magnitude_c` are all essentially unchanged from the model without `partner_belief`.

This is now confirmed under both risk-measure specifications: the partner-belief effect is not an artifact of how task difficulty is measured, and neither construction of the buffering interaction or the magnitude effect is disturbed by adding it.